In [1]:
import json
import re
import codecs
import urllib.request
import requests 
from bs4 import BeautifulSoup # for HTML parsing
import time # for sleep
from tqdm import tqdm # for progress tracker
import urllib3 # to disable SSL warnings 

# Disable the SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
import re

MONTH = (
    r"(?:January|February|March|April|May|June|July|August|"
    r"September|October|November|December)"
)
YEAR = r"(?:19|20)\d{2}"
DATE_POINT = rf"(?:(?:[0-3]?\d\s+)?{MONTH}\s+)?{YEAR}"
DATE_PERIOD = (
    rf"(?:{MONTH}\s+to\s+{MONTH}\s+{YEAR}|"
    rf"{DATE_POINT}(?:\s+to\s+{DATE_POINT})?)"
)
DATE_SUFFIX = re.compile(
    rf"\s*[:,]\s*(?:(?:between|(?:financial\s+)?year\s+ending)\s+)?"
    rf"{DATE_PERIOD}(?:\s+and\s+{DATE_PERIOD})*\s*$|"
    rf"\s*[:,]\s*mid-{YEAR}\s*$",
    re.IGNORECASE,
)


def derive_series(title):
    cleaned_title = title.strip()
    series = DATE_SUFFIX.sub("", cleaned_title).strip()
    return series or cleaned_title

In [ ]:
def extract_edition(html, title):
    heading = BeautifulSoup(html, "lxml").find("h1")
    if heading is None:
        return None

    heading_text = " ".join(heading.get_text(" ", strip=True).split())
    title_prefix = " ".join(title.split()).rstrip(":") + ":"
    if heading_text.startswith(title_prefix):
        return heading_text[len(title_prefix):].strip() or None

    suffix = DATE_SUFFIX.search(heading_text)
    if suffix is not None:
        return suffix.group().lstrip(" :,").strip() or None
    return None


def fetch_edition(doc_uri, title):
    response = requests.get(doc_uri, verify=False, timeout=30)
    response.raise_for_status()
    return extract_edition(response.text, title)

In [ ]:
from datetime import datetime


def mark_latest(records):
    newest_dates = {}
    dated_records = []
    for article in records:
        value = article.get('release_date')
        message = f"Invalid release_date for {article.get('doc_uri', 'unknown release')}: {value!r}"
        if not isinstance(value, str):
            raise ValueError(message)
        try:
            publication_date = datetime.fromisoformat(value.replace('Z', '+00:00')).date()
        except ValueError as error:
            raise ValueError(message) from error
        series = article['series']
        if series not in newest_dates or publication_date > newest_dates[series]:
            newest_dates[series] = publication_date
        dated_records.append((article, publication_date))

    for article, publication_date in dated_records:
        article['latest'] = publication_date == newest_dates[article['series']]

In [ ]:
from urllib.parse import urlsplit


def extract_themes(doc_uri):
    segments = [segment for segment in urlsplit(doc_uri).path.split('/') if segment]
    boundary = next((index for index, segment in enumerate(segments)
                     if segment in ('articles', 'bulletins', 'methodologies')), None)
    if boundary is None or not 1 <= boundary <= 3:
        raise ValueError(f"Unsupported theme hierarchy in doc_uri: {doc_uri}")

    hierarchy = segments[:boundary]
    return {name: hierarchy[index] if index < len(hierarchy) else None
            for index, name in enumerate(('theme', 'topic', 'subtopic'))}

In [2]:
# Opens and reads the JSON file gained from JT's data vis scraping tool 
with open('articles-and-dvcs.json', 'r') as f:
    results = json.load(f)
    
results.reverse()

In [3]:
pages = []

In [ ]:
for article in tqdm(results, desc="Processing articles"):

    if article['vis_urls']:
        article['edition'] = fetch_edition(article['doc_uri'], article['title'])
        article['vis'] = []
        for i, vis in enumerate(tqdm(article['vis_urls'], desc="Fetching visualizations", leave=False)):
            if 'datadownload.xlsx' in vis:
                continue
            url = "https://www.ons.gov.uk" + vis
            html = requests.get(url, verify=False).text

            # Only sleep between requests, not after the last one
            # if i < len(article['vis_urls']) - 1:
            #     time.sleep(0.5)

            soup = BeautifulSoup(html, "lxml")

# Most common: <meta name="template" content="...">
            meta = soup.find("meta", attrs={"name": "template"})
            template = meta.get("content") if meta else None
            article["vis"].append({"url": vis, "chart_type": template})

# Fallback: any meta where name/property contains "template"
            if template is None:
                meta = soup.find("meta", attrs={"name": lambda v: v and "template" in v.lower()}) \
                    or soup.find("meta", attrs={"property": lambda v: v and "template" in v.lower()})
                template = meta.get("content") if meta else None
            # print(template)

        for index, item in enumerate(pages):
            if item['doc_uri'] == article['doc_uri']:
                pages[index] = article
                break
        else:
            pages.append(article)

Processing articles:  13%|█▎        | 223/1674 [02:06<05:43,  4.23it/s]  C:\Users\rickas\AppData\Local\Temp\ipykernel_24064\4030569374.py:19: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
Processing articles: 100%|██████████| 1674/1674 [12:08<00:00,  2.30it/s]


In [5]:
# sort list by release date from newest to oldest
pages.sort(key=lambda x: x['release_date'], reverse=True)

In [ ]:
for article in pages:
    themes = extract_themes(article['doc_uri'])
    for field in themes:
        article.pop(field, None)
    article.update(themes)
    article['series'] = derive_series(article['title'])
    article.setdefault('edition', None)
    article.pop('vis_urls', None)

mark_latest(pages)

json_str = json.dumps(pages, indent=6)
with open("data.json", "w") as f:
    f.write(json_str)